In [2]:
%%bash
if test -f patents_db.sq3; then
  rm patents_db.sq3
fi
cat citations_short.csv | sqlite3 patents_db.sq3 ".mode csv" ".import /dev/stdin citations"
cat patents_short.csv | sqlite3 patents_db.sq3 ".mode csv" ".import /dev/stdin patents"

In [3]:
%load_ext sql
%sql sqlite:///patents_db.sq3


'Connected: @patents_db.sq3'

In [4]:
%%sql
SELECT * from patents as p;

 * sqlite:///patents_db.sq3
Done.


PATENT,APPYEAR,POSTATE
111,2000,CO
222,2001,TX
333,2002,CO
444,2003,PA
555,2004,CO
666,2005,PA


In [5]:
%%sql
SELECT * from citations;


 * sqlite:///patents_db.sq3
Done.


CITING,CITED
111,222
111,333
111,555
222,555
666,444
666,555


In [6]:
%%sql
SELECT * from citations c
JOIN patents p1
ON c.CITING==p1.PATENT


 * sqlite:///patents_db.sq3
Done.


CITING,CITED,PATENT,APPYEAR,POSTATE
111,222,111,2000,CO
111,333,111,2000,CO
111,555,111,2000,CO
222,555,222,2001,TX
666,444,666,2005,PA
666,555,666,2005,PA


In [7]:
%%sql
SELECT * from patents p1
JOIN citations c
ON p1.PATENT==c.CITING


 * sqlite:///patents_db.sq3
Done.


PATENT,APPYEAR,POSTATE,CITING,CITED
111,2000,CO,111,222
111,2000,CO,111,333
111,2000,CO,111,555
222,2001,TX,222,555
666,2005,PA,666,444
666,2005,PA,666,555


We'll stop before doing multiple joins.  But, use the below (and even the above) cells to play with SQL on a smaller dataset.

In [8]:
%%sql
SELECT
    c.CITING,
    p1.POSTATE AS CITING_POSTATE,
    c.CITED,
    p2.POSTATE AS CITED_POSTATE
FROM
    citations c
    JOIN patents p1 ON c.CITING = p1.PATENT
    JOIN patents p2 ON c.CITED = p2.PATENT;

 * sqlite:///patents_db.sq3
Done.


CITING,CITING_POSTATE,CITED,CITED_POSTATE
111,CO,222,TX
111,CO,333,CO
111,CO,555,CO
222,TX,555,CO
666,PA,444,PA
666,PA,555,CO


In [9]:
%%sql
SELECT p.PATENT AS PATENT, COUNT(c.CITED) AS CO_STATE_CITATIONS
FROM patents p
LEFT JOIN (
        SELECT c.CITING, c.CITED
        FROM citations c
        JOIN patents p1 ON c.CITING = p1.PATENT
        JOIN patents p2 ON c.CITED = p2.PATENT
        WHERE p1.POSTATE = p2.POSTATE
        ) c ON p.PATENT = c.CITED
GROUP BY p.PATENT;

 * sqlite:///patents_db.sq3
Done.


PATENT,CO_STATE_CITATIONS
111,0
222,0
333,1
444,1
555,1
666,0


In [10]:
%%sql
SELECT c.CITING, c.CITED
FROM citations c
JOIN patents p1 ON c.CITING = p1.PATENT
JOIN patents p2 ON c.CITED = p2.PATENT
WHERE p1.POSTATE = p2.POSTATE


 * sqlite:///patents_db.sq3
Done.


CITING,CITED
111,333
111,555
666,444


In [11]:
%%sql
SELECT p.*, COUNT(c.CITING) AS CO_STATE_CITATIONS 
FROM patents p
LEFT JOIN (
        SELECT c.CITING, c.CITED
        FROM citations c
        JOIN patents p1 ON c.CITING = p1.PATENT
        JOIN patents p2 ON c.CITED = p2.PATENT
        WHERE p1.POSTATE==p2.POSTATE
        ) c ON p.PATENT = c.CITED OR p.PATENT = c.CITING
GROUP BY p.PATENT
ORDER BY CO_STATE_CITATIONS DESC;

 * sqlite:///patents_db.sq3
Done.


PATENT,APPYEAR,POSTATE,CO_STATE_CITATIONS
111,2000,CO,2
666,2005,PA,1
555,2004,CO,1
444,2003,PA,1
333,2002,CO,1
222,2001,TX,0
